# Chapter 2: SGD and the MNIST Loss Function

Notes from reading "Deep Learning for Coders with fastai and PyTorch"

This notebook covers:
- How predictions work (weights, bias, parameters)
- The training loop
- Derivatives, slope, and gradients
- Learning rate
- Loss functions
- Sigmoid activation
- Building neural networks with nn.Sequential

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import tensor
import matplotlib.pyplot as plt
import numpy as np

## Part 1: The Simplest Prediction Function

The most basic model: multiply inputs by weights and sum.

In [ ]:
def predict(x, w):
    """Dot product: multiply element-wise, then sum to get a score."""
    return (x * w).sum()

# Simple example: 9 "pixels" (like a tiny 3x3 image flattened)
image = tensor([0., 0., 0., 1., 1., 1., 0., 0., 0.])  # vertical line in center
weights = tensor([0., 0., 0., 1., 1., 1., 0., 0., 0.])  # weights that "look for" vertical line

score = predict(image, weights)
print(f"Score: {score}")

### Why Weights Need Negative Values

With only positive weights (0s and 1s), we can't distinguish between "correct shape" and "everything lit up".

In [ ]:
# Problem: with only 0/1 weights
weights_bad = tensor([0., 0., 0., 1., 1., 1., 0., 0., 0.])

vertical_line = tensor([0., 0., 0., 1., 1., 1., 0., 0., 0.])
all_white = tensor([1., 1., 1., 1., 1., 1., 1., 1., 1.])

print(f"Vertical line score: {predict(vertical_line, weights_bad)}")
print(f"All white score: {predict(all_white, weights_bad)}")
print("Both score 3! Can't tell them apart.")

In [ ]:
# Solution: negative weights where we DON'T want ink
weights_good = tensor([-1., -1., -1., 1., 1., 1., -1., -1., -1.])
#                      "no ink"     "ink here"   "no ink"

print(f"Vertical line score: {predict(vertical_line, weights_good)}")
print(f"All white score: {predict(all_white, weights_good)}")
print("Now we can distinguish them!")

**Key insight:** 
- Positive weight = "I expect ink here" (evidence FOR)
- Negative weight = "I expect NO ink here" (evidence AGAINST)
- Zero weight = "I don't care about this pixel"

## Part 2: The Training Loop

The 7 steps to train any model:

1. **Initialize** weights (randomly)
2. **Predict** using current weights
3. **Calculate loss** (how wrong are we?)
4. **Calculate gradient** (which direction to adjust?)
5. **Step** (update weights based on gradient)
6. **Repeat** from step 2
7. **Stop** when good enough

In [ ]:
# Why random initialization works:
# The gradient tells us which way to go from ANY starting point

torch.manual_seed(42)
random_weights = torch.randn(9)  # random starting point
print(f"Random starting weights: {random_weights}")
print("\nThese will get better through training!")

### Epochs vs Iterations

- **Iteration**: One weight update (process one batch)
- **Epoch**: One complete pass through ALL training data

```
60,000 images, batch size 64:
1 epoch = 60,000 / 64 ≈ 937 iterations
```

Weights do NOT reset between epochs - they keep improving.

## Part 3: Derivatives and Slope

**Derivative = "If I nudge the input, how much does the output change?"**

This is the foundation of gradient descent.

In [ ]:
# The derivative of x² is 2x
# Let's verify this by actually measuring the slope

def f(x):
    return x ** 2

def measure_slope(x, nudge=0.0001):
    """Measure slope by computing rise/run with a tiny nudge."""
    rise = f(x + nudge) - f(x)
    run = nudge
    return rise / run

# Test at x = 3
x = 3
measured = measure_slope(x)
formula = 2 * x  # derivative of x² is 2x

print(f"At x = {x}:")
print(f"  Measured slope: {measured:.6f}")
print(f"  Formula (2x):   {formula}")

In [ ]:
# The smaller the nudge, the more accurate
x = 100

print(f"At x = {x}, derivative should be {2*x}\n")
for nudge in [1, 0.1, 0.01, 0.001, 0.0001, 0.00001]:
    slope = measure_slope(x, nudge)
    print(f"  Nudge {nudge:8} → slope = {slope:.6f}")

### What Slope Tells Us

**Slope = change in output / change in input**

| Slope | Meaning |
|-------|--------|
| Positive | Increasing input → increasing output |
| Negative | Increasing input → decreasing output |
| Large | Output is very sensitive |
| Small | Output barely reacts |
| Zero | Flat spot (minimum/maximum) |

In [ ]:
# Visualize the slope at different points on x²
x_vals = np.linspace(-3, 3, 100)
y_vals = x_vals ** 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot the curve
ax1.plot(x_vals, y_vals, 'b-', linewidth=2)
ax1.set_xlabel('x')
ax1.set_ylabel('f(x) = x²')
ax1.set_title('The function x²')
ax1.grid(True, alpha=0.3)

# Plot tangent lines at a few points
for x_point in [-2, 0, 2]:
    y_point = x_point ** 2
    slope = 2 * x_point  # derivative
    
    # Tangent line: y - y0 = slope * (x - x0)
    x_tangent = np.linspace(x_point - 1, x_point + 1, 10)
    y_tangent = slope * (x_tangent - x_point) + y_point
    
    ax1.plot(x_tangent, y_tangent, 'r--', alpha=0.7)
    ax1.plot(x_point, y_point, 'ro', markersize=8)
    ax1.annotate(f'slope={slope}', (x_point, y_point + 0.5))

# Plot the derivative itself
ax2.plot(x_vals, 2 * x_vals, 'g-', linewidth=2)
ax2.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax2.set_xlabel('x')
ax2.set_ylabel("f'(x) = 2x")
ax2.set_title('The derivative (slope at each point)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Calculus = "The Math of Infinitely Small Changes"

- **Differential calculus**: Studying infinitely small changes (derivatives, slopes)
- **Integral calculus**: Adding up infinitely many infinitely small pieces

For deep learning: PyTorch computes gradients automatically. You just need to understand what they *mean*.

## Part 4: Gradients with PyTorch

PyTorch tracks operations and computes gradients automatically.

In [ ]:
# requires_grad=True tells PyTorch to track this tensor
x = tensor(3.0, requires_grad=True)

# Do some computation
y = x ** 2

# Compute the gradient (derivative of y with respect to x)
y.backward()

print(f"x = {x.item()}")
print(f"y = x² = {y.item()}")
print(f"dy/dx = {x.grad.item()}")
print(f"Expected (2x) = {2 * x.item()}")

In [ ]:
# Gradient with multiple values
x = tensor([1., 2., 3., 4., 5.], requires_grad=True)
y = (x ** 2).sum()  # sum to get a scalar (required for backward)

y.backward()

print(f"x values: {x.data}")
print(f"Gradients: {x.grad}")
print(f"Expected (2x): {2 * x.data}")

## Part 5: Learning Rate

The gradient tells us *direction*. The learning rate controls *step size*.

```python
new_weight = old_weight - learning_rate * gradient
```

In [ ]:
# Scientific notation: cleaner than counting zeros
print("Common learning rates:")
print(f"  1e-2 = {1e-2}")
print(f"  1e-3 = {1e-3}")
print(f"  1e-4 = {1e-4}")
print(f"  1e-5 = {1e-5}")

In [ ]:
# Demonstrate why learning rate matters
# We'll try to find the minimum of f(x) = (x - 5)² 
# (minimum is at x = 5)

def train_step(x, lr):
    """One step of gradient descent."""
    x = tensor(x, requires_grad=True)
    loss = (x - 5) ** 2
    loss.backward()
    new_x = x.item() - lr * x.grad.item()
    return new_x, loss.item()

def train_loop(start, lr, steps=10):
    """Run multiple training steps."""
    x = start
    history = [(x, (x - 5) ** 2)]
    
    for _ in range(steps):
        x, loss = train_step(x, lr)
        history.append((x, loss))
    
    return history

# Try different learning rates
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, lr, title in zip(axes, [0.01, 0.1, 1.5], ['Too small (0.01)', 'Good (0.1)', 'Too big (1.5)']):
    history = train_loop(start=0.0, lr=lr, steps=20)
    xs, losses = zip(*history)
    
    ax.plot(losses, 'b-o', markersize=4)
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title(f'LR = {title}')
    ax.set_ylim(-1, 30)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Observations:**
- Too small: Converges but very slowly
- Good: Converges quickly to the minimum
- Too big: Overshoots and may never converge

## Part 6: PyTorch Datasets

A Dataset returns `(input, target)` tuples when indexed.

In [ ]:
# Creating a simple dataset with zip
inputs = [tensor([1., 2., 3.]), tensor([4., 5., 6.]), tensor([7., 8., 9.])]
targets = [0, 1, 1]

# zip pairs them up
dataset = list(zip(inputs, targets))

print("Dataset:")
for i, (x, y) in enumerate(dataset):
    print(f"  [{i}] input: {x}, target: {y}")

In [ ]:
# Indexing returns (x, y) tuple
x, y = dataset[0]
print(f"First example: x={x}, y={y}")

### The (x, y) Pattern Is Universal

| Task | x (input) | y (target) |
|------|-----------|------------|
| Image classification | Image | Label |
| Text generation | "The cat sat" | "on" |
| Translation | English | French |
| Q&A | Question + context | Answer |

## Part 7: Weights + Bias = Parameters

The linear equation: `y = w*x + b`

- `w` = weights (multipliers)
- `b` = bias (shifts output up/down)
- Both together = **parameters** (learnable values)

In [ ]:
# Simple linear model
def linear(x, w, b):
    return x @ w + b  # @ is matrix multiplication

# Example: 3 inputs, 1 output
x = tensor([1., 2., 3.])
w = tensor([[0.5], [0.3], [0.2]])  # shape: (3, 1)
b = tensor([0.1])

output = linear(x, w, b)
print(f"Input: {x}")
print(f"Output: {output}")
print(f"Manual: {1*0.5 + 2*0.3 + 3*0.2 + 0.1}")

### Matrix Multiplication (@)

Applies weights to entire batches at once.

In [ ]:
# Batch of 4 images, each with 784 pixels
batch = torch.randn(4, 784)
weights = torch.randn(784, 10)  # 10 outputs (digits 0-9)
bias = torch.randn(10)

# One operation processes all 4 images
output = batch @ weights + bias

print(f"Batch shape:   {batch.shape}")
print(f"Weights shape: {weights.shape}")
print(f"Output shape:  {output.shape}")
print("\n4 images → 4 sets of 10 predictions")

## Part 8: Loss Functions

Loss measures "how wrong is the model?"

In [ ]:
# Simple binary loss with torch.where
# torch.where(condition, value_if_true, value_if_false)

def binary_loss(predictions, targets):
    """
    If target is 1: loss = 1 - prediction (want prediction HIGH)
    If target is 0: loss = prediction (want prediction LOW)
    """
    return torch.where(targets == 1, 1 - predictions, predictions).mean()

# Example predictions (between 0 and 1)
predictions = tensor([0.9, 0.2, 0.8, 0.3])
targets = tensor([1, 0, 1, 0])  # binary: 1=yes, 0=no

loss = binary_loss(predictions, targets)
print("Predictions:", predictions.tolist())
print("Targets:    ", targets.tolist())
print(f"Loss: {loss.item():.4f}")
print("\n(Lower is better - these predictions are pretty good!)")

In [ ]:
# Bad predictions
bad_predictions = tensor([0.1, 0.9, 0.2, 0.8])  # opposite of what we want
bad_loss = binary_loss(bad_predictions, targets)

print("Bad predictions:", bad_predictions.tolist())
print("Targets:        ", targets.tolist())
print(f"Loss: {bad_loss.item():.4f}")
print("\n(Higher loss = worse predictions)")

### Binary vs Multi-class

| Type | Target Values | Use Case |
|------|---------------|----------|
| Binary | 0 or 1 | Is it a 3? Spam/not spam |
| Multi-class | 0, 1, 2, ... N | Which digit? Which category? |

Text generation is **massive multi-class**: predicting from ~50,000 tokens!

## Part 9: Sigmoid Activation

Squashes any number into 0-1 range (for probabilities).

In [ ]:
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

# Test on various inputs
test_values = tensor([-10., -3., 0., 3., 10.])
outputs = sigmoid(test_values)

print("Input → Sigmoid output")
for inp, out in zip(test_values.tolist(), outputs.tolist()):
    print(f"  {inp:6.1f} → {out:.6f}")

In [ ]:
# Visualize the S-curve
x = torch.linspace(-8, 8, 100)
y = sigmoid(x)

plt.figure(figsize=(10, 5))
plt.plot(x.numpy(), y.numpy(), 'b-', linewidth=2)
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
plt.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
plt.axhline(y=1, color='gray', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Input (raw model output / logits)')
plt.ylabel('Output (probability)')
plt.title('Sigmoid: Squashes any value to 0-1')
plt.grid(True, alpha=0.3)

# Annotate
plt.annotate('Confident NO\n(near 0)', xy=(-6, 0.05), fontsize=10)
plt.annotate('Uncertain\n(0.5)', xy=(0.5, 0.55), fontsize=10)
plt.annotate('Confident YES\n(near 1)', xy=(4, 0.9), fontsize=10)

plt.show()

### Why Sigmoid (not just clipping)?

- **Smooth**: Gradients can flow through
- **Differentiable**: Can compute derivatives everywhere
- Hard clipping (`max(0, min(1, x))`) has zero gradient at extremes → training stops

## Part 10: Building Neural Networks with nn.Sequential

Stack layers to create more complex functions.

In [ ]:
# A simple neural network for MNIST
simple_net = nn.Sequential(
    nn.Linear(784, 30),    # 784 inputs → 30 hidden neurons
    nn.ReLU(),             # Activation function
    nn.Linear(30, 10),     # 30 hidden → 10 outputs (digits 0-9)
)

print(simple_net)

In [ ]:
# Test with a fake batch
fake_batch = torch.randn(8, 784)  # 8 images, 784 pixels each
output = simple_net(fake_batch)

print(f"Input shape:  {fake_batch.shape}")
print(f"Output shape: {output.shape}")
print(f"\nFirst image's 10 scores: {output[0].data}")

### ReLU: The Most Common Activation

`ReLU(x) = max(0, x)`

- Negative → 0
- Positive → unchanged

In [ ]:
# Visualize ReLU
x = torch.linspace(-5, 5, 100)
y_relu = F.relu(x)
y_sigmoid = torch.sigmoid(x)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(x.numpy(), y_relu.numpy(), 'b-', linewidth=2)
ax1.set_xlabel('Input')
ax1.set_ylabel('Output')
ax1.set_title('ReLU: max(0, x)')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax1.axvline(x=0, color='gray', linestyle='-', alpha=0.3)

ax2.plot(x.numpy(), y_sigmoid.numpy(), 'r-', linewidth=2)
ax2.set_xlabel('Input')
ax2.set_ylabel('Output')
ax2.set_title('Sigmoid: 1/(1+e^-x)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Why Activation Functions?

Without them, stacking linear layers is useless:

```
Linear → Linear = still just Linear
(multiple linear transforms collapse into one)
```

With activation:

```
Linear → ReLU → Linear → ReLU → Linear
(can approximate ANY function!)
```

This is the **Universal Approximation Theorem**: A neural network with enough hidden units can approximate any continuous function.

## Part 11: Putting It All Together

A complete training loop on synthetic data.

In [ ]:
# Create synthetic data: learn to classify points above/below a line
torch.manual_seed(42)

# Generate 200 random 2D points
n_samples = 200
X = torch.randn(n_samples, 2)  # 200 points, 2 features each

# Label: 1 if x1 + x2 > 0, else 0
y = (X[:, 0] + X[:, 1] > 0).float().unsqueeze(1)  # shape: (200, 1)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFirst 5 examples:")
for i in range(5):
    print(f"  Point {X[i].tolist()} → label {int(y[i].item())}")

In [ ]:
# Visualize the data
plt.figure(figsize=(8, 6))
colors = ['red' if label == 0 else 'blue' for label in y]
plt.scatter(X[:, 0].numpy(), X[:, 1].numpy(), c=colors, alpha=0.6)
plt.axline((0, 0), slope=-1, color='gray', linestyle='--', label='x1 + x2 = 0')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Blue = above line (1), Red = below line (0)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Build the model
model = nn.Sequential(
    nn.Linear(2, 8),     # 2 inputs → 8 hidden
    nn.ReLU(),
    nn.Linear(8, 1),     # 8 hidden → 1 output
    nn.Sigmoid()         # Convert to probability
)

# Loss function and optimizer
loss_fn = nn.BCELoss()  # Binary Cross Entropy
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

print("Model:")
print(model)

In [ ]:
# Training loop
losses = []
n_epochs = 100

for epoch in range(n_epochs):
    # Forward pass
    predictions = model(X)
    loss = loss_fn(predictions, y)
    
    # Backward pass
    optimizer.zero_grad()  # Clear old gradients
    loss.backward()        # Compute new gradients
    optimizer.step()       # Update weights
    
    losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        # Calculate accuracy
        with torch.no_grad():
            preds = (model(X) > 0.5).float()
            accuracy = (preds == y).float().mean()
        print(f"Epoch {epoch+1:3d}: loss = {loss.item():.4f}, accuracy = {accuracy.item():.2%}")

In [ ]:
# Plot training progress
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Visualize what the model learned
# Create a grid of points to see the decision boundary

x_range = torch.linspace(-3, 3, 100)
y_range = torch.linspace(-3, 3, 100)
xx, yy = torch.meshgrid(x_range, y_range, indexing='ij')
grid = torch.stack([xx.flatten(), yy.flatten()], dim=1)

with torch.no_grad():
    grid_preds = model(grid).reshape(100, 100)

plt.figure(figsize=(10, 8))
plt.contourf(xx.numpy(), yy.numpy(), grid_preds.numpy(), levels=20, cmap='RdBu', alpha=0.7)
plt.colorbar(label='Model confidence')
plt.scatter(X[:, 0].numpy(), X[:, 1].numpy(), c=colors, edgecolors='white', s=50)
plt.contour(xx.numpy(), yy.numpy(), grid_preds.numpy(), levels=[0.5], colors='black', linewidths=2)
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Model Decision Boundary (black line = 50% confidence)')
plt.show()

## Key Takeaways

1. **Weights can be negative** - needed to distinguish shapes from "all white"

2. **Training loop**: Initialize → Predict → Loss → Gradient → Step → Repeat

3. **Derivative = slope = sensitivity** - "If I nudge input, how much does output change?"

4. **Learning rate** scales gradient to reasonable step size (typically 0.001 - 0.1)

5. **Dataset** returns `(input, target)` tuples - universal pattern for all ML

6. **Parameters = weights + bias** in the equation `y = wx + b`

7. **Matrix multiplication (@)** processes batches efficiently

8. **Sigmoid** squashes values to 0-1 for probabilities

9. **Activation functions** (ReLU, Sigmoid) make deep networks powerful

10. **Universal Approximation Theorem**: Deep enough networks can learn any function

## Next Steps

- Apply this to real MNIST data
- Experiment with different architectures (more layers, different widths)
- Try different learning rates and see overshoot vs slow convergence
- Explore validation loss to detect overfitting